In [41]:
EMA_path = "./../data/datasets/EMA.json"
JAPAN_path = "./../data/datasets/JAPAN.json"
SWISSMEDIC_path = "./../data/datasets/SWISSMEDIC.json"
AUSTRALIA_path = "./../data/datasets/AUSTRALIA.json"
FDA_path = "./../data/datasets/FDA.json"
HEALTHCANADA_path = "./../data/datasets/HEALTHCANADA.json"

In [61]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.colors as mcolors
import seaborn as sns
import json
import numpy as np
from matplotlib.ticker import FixedLocator, ScalarFormatter
import math
import os

In [ ]:
# -----------------------------
# Paths zu den JSON-Dateien
# -----------------------------
agency_paths = {
    "EMA": EMA_path,
    "FDA": FDA_path,
    "SWISSMEDIC": SWISSMEDIC_path,
    "JAPAN": JAPAN_path,
    "AUSTRALIA": AUSTRALIA_path,
    "HEALTHCANADA": HEALTHCANADA_path
}

# Agencies mit numerischen Dict-Keys (0,1,2,...)
DICT_NUMERIC_KEY_AGENCIES = {"FDA", "HEALTHCANADA"}

# -----------------------------
# JSON Loader
# -----------------------------
def load_agency_json(path: str, agency: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Fall 1: Dict mit Records als Values (FDA, HC, EMA, Japan, Australia, ...)
    if isinstance(data, dict):
        df = pd.DataFrame.from_dict(data, orient="index")
        df = df.reset_index().rename(columns={"index": "Record_key"})

        # numerische Keys bei FDA / HealthCanada entfernen
        if agency in DICT_NUMERIC_KEY_AGENCIES:
            df = df.drop(columns=["Record_key"])

        return df

    # Fall 2: Liste von Records (Fallback)
    if isinstance(data, list):
        return pd.DataFrame(data)

    raise ValueError(f"Unerwartetes JSON-Format für {agency}: {type(data)}")

# -----------------------------
# Helper für Identifier-Zählung
# -----------------------------
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def cleaned_series(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.mask(x.str.lower().isin(PLACEHOLDERS))
    return x

def agency_identifier_count(df: pd.DataFrame, agency: str) -> int:
    """
    Zählt den passenden Identifier pro Agency:
    - Japan: Anzahl PDF-Records (Record_key)
    - alle anderen: unique Marketing_authorisation_number (ohne Platzhalter)
    """
    if agency == "JAPAN":
        return df["Record_key"].nunique()

    s = cleaned_series(df["Marketing_authorisation_number"])
    return s.nunique(dropna=True)

# -----------------------------
# Overall-DataFrame
# (nur für Status-Auswertungen!)
# -----------------------------
df_overall = pd.concat(
    [
        df_fda,
        df_healthcanada,
        df_ema,
        df_swissmedic,
        df_japan,
        df_australia
    ],
    ignore_index=True
)


# Results 1. Dataset Characteristics

Numbers per application

In [62]:
agencies = {
    "FDA": df_fda,
    "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Japan": df_japan,
    "Australia": df_australia,
}

# Anzahl Records pro Agency
rows = []
total_records = sum(len(df) for df in agencies.values())

for name, df in agencies.items():
    n = len(df)
    pct = round(n / total_records * 100, 2) if total_records else 0.0
    rows.append({
        "Agency": name,
        "n_records": n,
        "%_of_overall_records": pct
    })

record_summary = pd.DataFrame(rows)

print("Record counts per agency (based on JSON entries)")
display(record_summary)


Record counts per agency (based on JSON entries)


,Agency,n_records,%_of_overall_records
0,FDA,28288,65.04
1,Health Canada,11523,26.49
2,EMA,1993,4.58
3,Swissmedic,234,0.54
4,Japan,408,0.94
5,Australia,1050,2.41


Numbers per drug (unique Marketing_authorisation_number)

In [63]:
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

agencies = {
    "FDA": df_fda,
    "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Japan": df_japan,
    "Australia": df_australia,
}

def clean_ma_series(s: pd.Series) -> set:
    return set(
        s.astype("string")
         .str.strip()
         .str.lower()
         .mask(lambda x: x.isin(PLACEHOLDERS))
         .dropna()
         .unique()
    )

# ------------------------------------------------
# Collect agency-specific identifiers
# ------------------------------------------------
agency_ids = {}
overall_ids = 0

for name, df in agencies.items():

    # Japan: count PDFs
    if name == "Japan":
        n_ids = df["Record_key"].nunique()
        agency_ids[name] = n_ids
        overall_ids += n_ids
        continue

    # All other agencies: unique MA numbers
    ma_set = clean_ma_series(df["Marketing_authorisation_number"])
    n_ids = len(ma_set)
    agency_ids[name] = n_ids
    overall_ids += n_ids

# ------------------------------------------------
# Summary table
# ------------------------------------------------
rows = []
for name, n in agency_ids.items():
    pct = round(n / overall_ids * 100, 2) if overall_ids else 0.0
    rows.append({
        "Agency": name,
        "n_unique_identifiers": n,
        "%_of_overall": pct
    })

ma_summary = pd.DataFrame(rows)

print("Unique identifiers per agency (MA numbers; Japan = PDFs)")
display(ma_summary)


Unique identifiers per agency (MA numbers; Japan = PDFs)


,Agency,n_unique_identifiers,%_of_overall
0,FDA,28288,66.16
1,Health Canada,11523,26.95
2,EMA,1459,3.41
3,Swissmedic,196,0.46
4,Japan,408,0.95
5,Australia,882,2.06


Table 1: Key characteristics of approvals

In [ ]:
# ============================================================
# Imports
# ============================================================
import pandas as pd
from IPython.display import display

# ============================================================
# Constants & helpers
# ============================================================
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

DRUG_CLASS_ORDER = [
    "small molecule",
    "biologics",
    "cell and gene therapy",
    "peptides and proteins",
    "vaccine",
    "not reported",
    "other",
]

def norm_series(s: pd.Series) -> pd.Series:
    """Strip whitespace, lower-case, keep NaN."""
    return s.astype("string").str.strip().str.lower()

# ============================================================
# Decision distribution (per record) + consolidated
# ============================================================
def decision_distribution(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)

    decision = (
        df.get("Decision", pd.Series([pd.NA] * n))
          .astype("string")
          .str.strip()
          .str.lower()
    )

    decision = decision.mask(decision.isin(PLACEHOLDERS), "not reported")

    counts = decision.value_counts(dropna=False)
    dist = counts.reset_index()
    dist.columns = ["Decision", "n"]
    dist["%"] = (dist["n"] / n * 100).round(2) if n else 0.0

    consolidated_mask = decision.isin({
        "approved",
        "conditional marketing authorisation",
        "conditional marketing authorization",
    })
    consolidated_n = int(consolidated_mask.sum())
    consolidated_pct = round(consolidated_n / n * 100, 2) if n else 0.0

    consolidated_row = pd.DataFrame([{
        "Decision": "consolidated (approved + conditional marketing authorisation)",
        "n": consolidated_n,
        "%": consolidated_pct
    }])

    return pd.concat([dist, consolidated_row], ignore_index=True)

# ============================================================
# Decision per unique Marketing Authorisation Number (= drugs) + consolidated
# ============================================================
def decision_per_unique_ma(df: pd.DataFrame) -> pd.DataFrame:
    if "Marketing_authorisation_number" not in df.columns:
        return pd.DataFrame(columns=["Decision", "n", "%"])

    tmp = df[["Marketing_authorisation_number", "Decision"]].copy()

    tmp["ma"] = (
        tmp["Marketing_authorisation_number"]
        .astype("string")
        .str.strip()
        .str.lower()
    )
    tmp = tmp[~tmp["ma"].isin(PLACEHOLDERS)]
    tmp = tmp.dropna(subset=["ma"])

    if tmp.empty:
        return pd.DataFrame(columns=["Decision", "n", "%"])

    tmp["decision"] = (
        tmp["Decision"]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS), "not reported")
    )

    tmp_unique = tmp.drop_duplicates(subset=["ma"])
    n_drugs = len(tmp_unique)

    counts = tmp_unique["decision"].value_counts(dropna=False)
    dist = counts.reset_index()
    dist.columns = ["Decision", "n"]
    dist["%"] = (dist["n"] / n_drugs * 100).round(2) if n_drugs else 0.0

    consolidated_mask = tmp_unique["decision"].isin({
        "approved",
        "conditional marketing authorisation",
        "conditional marketing authorization",
    })
    consolidated_n = int(consolidated_mask.sum())
    consolidated_pct = round(consolidated_n / n_drugs * 100, 2) if n_drugs else 0.0

    consolidated_row = pd.DataFrame([{
        "Decision": "consolidated (approved + conditional marketing authorisation)",
        "n": consolidated_n,
        "%": consolidated_pct
    }])

    return pd.concat([dist, consolidated_row], ignore_index=True)

# ============================================================
# Drug class summary (record-based, exact categories, case-insensitive)
# ============================================================
def bucket_drug_class(x) -> str:
    if pd.isna(x):
        return "not reported"

    t = str(x).strip().lower()
    if t in PLACEHOLDERS:
        return "not reported"

    if t in DRUG_CLASS_ORDER:
        return t

    return "other"

def drug_class_summary(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    s = df.get("Drug_class", pd.Series([pd.NA] * n)).map(bucket_drug_class)
    counts = s.value_counts()

    rows = []
    for c in DRUG_CLASS_ORDER:
        k = int(counts.get(c, 0))
        rows.append({
            "Drug class": c,
            "n": k,
            "%": round(k / n * 100, 2) if n else 0.0
        })
    return pd.DataFrame(rows)

# ============================================================
# Therapeutic area composition (Top 5, mention-based)  [as in your original]
# ============================================================
def therapeutic_area_top5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = norm_series(df[col])
    s = s.mask(s.isin(PLACEHOLDERS)).dropna()

    if s.empty:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    counts = exploded.value_counts().head(5)
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Therapeutic area", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out

# ============================================================
# OVERALL helpers
# ============================================================
def _overall_decision_distribution(agencies: dict) -> pd.DataFrame:
    df_all = pd.concat(list(agencies.values()), ignore_index=True)
    return decision_distribution(df_all)

def _overall_decision_per_unique_ma(agencies: dict) -> pd.DataFrame:
    # overall per drug = unique MA across all agencies (duplicates across agencies collapsed)
    df_all = pd.concat(list(agencies.values()), ignore_index=True)
    return decision_per_unique_ma(df_all)

def _overall_drug_class_summary(agencies: dict) -> pd.DataFrame:
    df_all = pd.concat(list(agencies.values()), ignore_index=True)
    return drug_class_summary(df_all)

def _overall_therapeutic_area_top5(agencies: dict) -> pd.DataFrame:
    df_all = pd.concat(list(agencies.values()), ignore_index=True)
    return therapeutic_area_top5(df_all)

# ============================================================
# Run for all agencies + OVERALL (one cell)
# ============================================================
agencies = {
    "FDA": df_fda,
    "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Japan": df_japan,
    "Australia": df_australia,
}

# ---- Per agency
for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name} | Records: {len(df)}")

    print("\nDecision distribution (per record)")
    display(decision_distribution(df))

    print("\nDecision per unique Marketing Authorisation Number (number of drugs)")
    display(decision_per_unique_ma(df))

    print("\nDrug classes (per record)")
    display(drug_class_summary(df))

    print("\nTherapeutic area composition (Top 5, mention-based)")
    display(therapeutic_area_top5(df))

# ---- OVERALL
print("\n" + "=" * 70)
print("OVERALL | Records:", sum(len(df) for df in agencies.values()))

print("\nDecision distribution (per record) — OVERALL")
display(_overall_decision_distribution(agencies))

print("\nDecision per unique Marketing Authorisation Number (number of drugs) — OVERALL")
display(_overall_decision_per_unique_ma(agencies))

print("\nDrug classes (per record) — OVERALL")
display(_overall_drug_class_summary(agencies))

print("\nTherapeutic area composition (Top 5, mention-based) — OVERALL")
display(_overall_therapeutic_area_top5(agencies))



FDA | Records: 28288

Decision distribution (per record)


,Decision,n,%
0,approved,24309,85.93
1,<NA>,2920,10.32
2,conditional marketing authorisation,1059,3.74
3,consolidated (approved + conditional marketing...,25368,89.68



Decision per unique Marketing Authorisation Number (number of drugs)


,Decision,n,%
0,approved,24309,85.93
1,<NA>,2920,10.32
2,conditional marketing authorisation,1059,3.74
3,consolidated (approved + conditional marketing...,25368,89.68



Drug classes


...

,Decision,n,%
0,approved,870,98.64
1,withdrawn,9,1.02
2,rejected,3,0.34
3,consolidated (approved + conditional marketing...,870,98.64



Drug classes


,Drug class,n,%
0,small molecule,540,51.43
1,biologics,336,32.00
2,cell and gene therapy,4,0.38
3,peptides and proteins,60,5.71
4,vaccine,93,8.86
5,not reported,0,0.00
6,other,17,1.62



Therapeutic area composition (Top 5)


,Therapeutic area,n,%
0,neoplasms,281,19.09
1,infectious and parasitic diseases,183,12.43
2,diseases of the blood and blood-forming organs,145,9.85
3,diseases of the respiratory system,133,9.04
4,"endocrine, nutritional, and metabolic diseases",125,8.49
